In [1]:
# clinical deterioration: new onset among patients with initially normal vital signs (first 15 minutes)
# tachycardia: HR > 110
# hypotension: MAP < 65
# hypoxia: SpO2 < 90

In [2]:
import os
import pandas as pd
import pickle
import numpy as np

In [3]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/mcmed'

LABEL

In [4]:
numerics_df = pd.read_csv(os.path.join(DATA_DIR, 'numerics.csv'))

In [5]:
HR_df = numerics_df[(numerics_df['Measure'] == 'HR') & (numerics_df['Value'] > 110)]
MAP_df = numerics_df[(numerics_df['Measure'] == 'MAP') & (numerics_df['Value'] < 65)]
SpO2_df = numerics_df[(numerics_df['Measure'] == 'SpO2') & (numerics_df['Value'] < 90)]

In [ ]:
decomp_df = pd.concat([HR_df, MAP_df, SpO2_df])
decomp_df = decomp_df[['CSN', 'Time']].groupby('CSN').min().reset_index().rename(columns={'Time': 'Criteria_time'})

In [12]:
decomp_df['CSN'].nunique()

34301

CREATE DATASET

In [13]:
from scipy.stats import gaussian_kde

In [23]:
demo_df = pd.read_csv(os.path.join(DATA_DIR, 'demo.csv'))

In [36]:
df = demo_df[['CSN', 'Arrival_time']].merge(decomp_df, on='CSN', how='left')

In [37]:
df['Label'] = ~df['Criteria_time'].isna()

In [38]:
df['Criteria_time'] = pd.to_datetime(df['Criteria_time'])
df['Arrival_time'] = pd.to_datetime(df['Arrival_time'])
df['Time'] = (df['Criteria_time'] - df['Arrival_time']).dt.total_seconds() / 3600

In [39]:
neg_df = df[df['Label'] == False]
pos_df = df[df['Label'] == True]

In [ ]:
original = (pos_df['Time'].values)
kde = gaussian_kde(original)
times = kde.resample(size=len(neg_df)*2).flatten()
times = times[times > 0]
neg_df.loc[:, 'Time'] = times[:len(neg_df)]

In [58]:
label_df = pd.concat([neg_df, pos_df])
df = (df.drop(columns='Time')).merge(label_df[['CSN', 'Time']], on='CSN', how='left')

In [59]:
df.to_csv(os.path.join(DATA_DIR, "decomp_label.csv"), index=False)

ADD CRITERIA

In [ ]:
with open('/home/mkeoliya/projects/CAREBench/data/mcmed/numerics_buckets.pkl', 'rb') as f:
        buckets = pickle.load(f)

In [ ]:
hr_bucket = np.searchsorted(buckets['HR'], 110, side='right')
map_bucket = np.searchsorted(buckets['MAP'], 65, side='right')
SpO2_bucket = np.searchsorted(buckets['SpO2'], 90, side='right')
print(hr_bucket, map_bucket, SpO2_bucket)

In [26]:
hr_buckets = [10, 11]
map_bucket = [1]
SpO2_buckets = [1]

In [27]:
hr_criteria, map_criteria, spo2_criteria = [], [], []
for inds, event, criteria in [(hr_buckets, 'HR', hr_criteria), (map_bucket, 'MAP', map_criteria), (SpO2_buckets, 'SpO2', spo2_criteria)]:
    for ind in inds:
        bucket = buckets[event]
        if ind == len(bucket):
            eventval = f"{event}|{bucket[ind-1]}-"
        else:
            eventval = f"{event}|{bucket[ind-1]}-{bucket[ind]}"
        criteria.append(eventval)
hr_criteria, map_criteria, spo2_criteria

(['HR|106.0667-268.0', 'HR|268.0-'], ['MAP|1.0-75.6667'], ['SpO2|0.0-94.7907'])